# WHFDL — SHAP & LIME Explainability

## Imports & Settings

In [ ]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

# ---- standard library ----
import os
import random
import warnings

# ---- data & numeric ----
import numpy as np
import pandas as pd

# ---- plotting ----
import matplotlib.pyplot as plt
from matplotlib import rcParams

# ---- deep learning ----
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- scikit-learn ----
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, classification_report

# ---- explainability ----
import shap
from lime.lime_tabular import LimeTabularExplainer

# =========================================================
# SETTINGS
# =========================================================

warnings.filterwarnings("ignore")
rcParams['font.family'] = 'Times New Roman'

seed = 42
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
# Input data lives under DATA_DIR (override with: export DATA_DIR=/path/to/data)
from pathlib import Path
DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))

results_dir = "../results/Explainability"
os.makedirs(results_dir, exist_ok=True)

## Load Data

In [ ]:
# =========================================================
# LOAD DATA
# =========================================================

DATA_PATH = DATA_DIR / "mRMR_30f.csv"

data = pd.read_csv(DATA_PATH)

X = data.drop("label", axis=1)
y = data["label"]

feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# =========================================================
# SCALE & CONVERT TO TENSORS
# =========================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

num_classes = len(np.unique(y))
input_dim = X_train_scaled.shape[1]

print(f"{input_dim} features, {num_classes} classes, "
      f"{len(X_train_scaled)} train / {len(X_test_scaled)} test samples")

## WHFDL Model

In [ ]:
# =========================================================
# FUZZY MEMBERSHIP LAYER
# =========================================================

class FuzzyMembership(nn.Module):
    """Gaussian fuzzy membership layer, K-Means initialized then fine-tuned by backprop."""

    def __init__(self, input_dim, membership_units):
        super().__init__()
        self.membership_units = membership_units
        self.mu = nn.Parameter(torch.randn(membership_units, input_dim))
        self.log_sigma = nn.Parameter(torch.zeros(membership_units, input_dim))

    @torch.no_grad()
    def init_from_data(self, X):
        """K-Means initialization of (mu, sigma)."""
        X_np = X.detach().cpu().numpy()
        n_clusters = min(self.membership_units, len(X_np))
        km = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
        labels = km.fit_predict(X_np)

        centers = torch.zeros_like(self.mu)
        sigmas = torch.ones_like(self.mu)

        for k in range(self.membership_units):
            k_idx = k % n_clusters
            cluster_pts = X[torch.tensor(labels == k_idx)]
            centers[k] = torch.tensor(km.cluster_centers_[k_idx], dtype=torch.float32)
            if len(cluster_pts) > 1:
                sigmas[k] = cluster_pts.std(dim=0) + 1e-3
            else:
                sigmas[k] = X.std(dim=0) + 1e-3

        self.mu.copy_(centers)
        self.log_sigma.copy_(torch.log(sigmas))

    def forward(self, x):
        x = x.unsqueeze(1)
        sigma = torch.exp(self.log_sigma) + 1e-6
        membership = torch.exp(-((x - self.mu) ** 2) / (sigma ** 2))
        return membership

# =========================================================
# WHFDL CLASSIFIER
# =========================================================

class WHFDL(nn.Module):
    """Hierarchical Fused Fuzzy Deep Neural Network (WHFDL classifier)."""

    def __init__(self, input_size, membership_units=3, hidden_units=128,
                 dropout_rate=0.3, num_classes=2):
        super().__init__()

        # ---- fuzzy branch ----
        self.fuzzy = FuzzyMembership(input_size, membership_units)
        self.fuzzy_proj = nn.Linear(membership_units, hidden_units)

        # ---- deep branch ----
        self.deep_fc = nn.Linear(input_size, hidden_units)
        bound = 1.0 / np.sqrt(input_size)  # fan-in uniform init
        nn.init.uniform_(self.deep_fc.weight, -bound, bound)
        nn.init.zeros_(self.deep_fc.bias)

        # ---- fusion block: trainable per-node weights for each branch ----
        self.w_d = nn.Parameter(torch.ones(hidden_units))
        self.w_f = nn.Parameter(torch.ones(hidden_units))
        self.fusion_bias = nn.Parameter(torch.zeros(hidden_units))
        self.fusion_fc = nn.Linear(hidden_units, hidden_units)
        self.dropout = nn.Dropout(dropout_rate)

        # ---- output layer ----
        self.classifier = nn.Linear(hidden_units, num_classes)

    def forward(self, x):
        membership = self.fuzzy(x)
        fuzzy_rule = torch.prod(membership, dim=-1)
        o_f = torch.sigmoid(self.fuzzy_proj(fuzzy_rule))

        o_d = torch.sigmoid(self.deep_fc(x))

        fused = self.w_d * o_d + self.w_f * o_f + self.fusion_bias
        fused = torch.sigmoid(fused)
        fused = F.relu(self.fusion_fc(fused))
        fused = self.dropout(fused)

        out = self.classifier(fused)
        return out

## Train WHFDL

In [ ]:
# =========================================================
# BUILD MODEL & TRAINING SETUP
# =========================================================

model = WHFDL(input_size=input_dim, num_classes=num_classes).to(device)
model.fuzzy.init_from_data(X_train_tensor)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 30
BATCH_SIZE = 32

train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# =========================================================
# TRAINING LOOP
# =========================================================

model.train()
for epoch in range(EPOCHS):
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

# =========================================================
# EVALUATE ON TEST SET
# =========================================================

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor.to(device))
    test_preds = torch.argmax(test_outputs, dim=1).cpu().numpy()

print("Test accuracy:", accuracy_score(y_test, test_preds))
print(classification_report(y_test, test_preds))

## Prediction Wrapper

In [ ]:
# =========================================================
# PREDICTION WRAPPER (needed by SHAP / LIME)
# =========================================================

def predict_proba(X_array):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(np.asarray(X_array), dtype=torch.float32).to(device)
        outputs = model(X_tensor)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
    return probs

## LIME 

In [ ]:
# =========================================================
# LIME LOCAL EXPLANATION
# =========================================================

LIME_INSTANCE_IDX = 0
LIME_NUM_FEATURES = 10

explainer_lime = LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=feature_names,
    class_names=["Negative", "Positive"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

instance = X_test_scaled[LIME_INSTANCE_IDX]
true_label = y_test.values[LIME_INSTANCE_IDX]
pred_prob = predict_proba(instance.reshape(1, -1))[0]

print(f"Instance #{LIME_INSTANCE_IDX} | true label: {true_label} | "
      f"predicted P(Positive) = {pred_prob[1]:.4f}")

exp = explainer_lime.explain_instance(
    instance,
    predict_proba,
    num_features=LIME_NUM_FEATURES
)

# =========================================================
# PLOT
# =========================================================

lime_df = pd.DataFrame(exp.as_list(), columns=["feature", "contribution"])

# sort by absolute importance, ascending (largest bar ends up on top
# with plt.barh's default bottom-to-top category order)
lime_df["abs_contribution"] = lime_df["contribution"].abs()
lime_df = lime_df.sort_values("abs_contribution", ascending=True)

colors = ["#298c8c" if c > 0 else "#a00000" for c in lime_df["contribution"]]
max_abs = np.abs(lime_df["contribution"]).max()

plt.figure(figsize=(20, 10))

plt.barh(
    lime_df["feature"],
    lime_df["contribution"],
    color=colors
)

plt.yticks(fontsize=15)

# zero line
plt.axvline(x=0, color="black", linewidth=1)

# make x-axis symmetric around zero
plt.xlim(-max_abs * 1.1, max_abs * 1.1)

plt.xlabel("LIME Contribution", fontsize=17, fontweight="bold")
plt.ylabel("Feature", fontsize=17, fontweight="bold")
plt.title("LIME local Feature Importance", fontsize=20, fontweight="bold")

plt.grid(axis="x", alpha=0.2)

plt.tight_layout()

plt.savefig(
    f"{results_dir}/LIME_local_feature_importance.png",
    dpi=1500,
    bbox_inches="tight"
)
plt.show()

## SHAP 

In [ ]:
# =========================================================
# SHAP GLOBAL EXPLANATION
# =========================================================

SHAP_BACKGROUND_SIZE = 50
SHAP_SAMPLE_SIZE = 100
SHAP_NSAMPLES = 100
SHAP_MAX_DISPLAY = 10

background = shap.kmeans(X_train_scaled, min(SHAP_BACKGROUND_SIZE, len(X_train_scaled)))
explainer_shap = shap.KernelExplainer(predict_proba, background)

X_sample = X_test_scaled[:min(SHAP_SAMPLE_SIZE, len(X_test_scaled))]
shap_values_raw = explainer_shap.shap_values(X_sample, nsamples=SHAP_NSAMPLES)

# =========================================================
# DIMENSION HANDLING
# =========================================================
# shap_values_raw's shape depends on the SHAP version and whether the
# model returns one array per class or a single stacked array — this
# normalizes it down to a single (n_samples, n_features) array for the
# "Positive" class.

if isinstance(shap_values_raw, list):
    shap_vals = shap_values_raw[1]
else:
    shap_vals = shap_values_raw

shap_vals = np.array(shap_vals)

if shap_vals.ndim == 3:
    if shap_vals.shape[2] == 2:
        shap_vals = shap_vals[:, :, 1]
    else:
        shap_vals = shap_vals[:, :, 0]
elif shap_vals.ndim == 2:
    pass
else:
    raise ValueError(f"Unexpected SHAP dimensions: {shap_vals.shape}")

if shap_vals.shape[0] != X_sample.shape[0]:
    shap_vals = shap_vals[:X_sample.shape[0]]

if shap_vals.shape[1] != X_sample.shape[1]:
    raise ValueError(
        f"Feature mismatch: SHAP={shap_vals.shape[1]}, DATA={X_sample.shape[1]}"
    )

X_sample_df = pd.DataFrame(X_sample, columns=feature_names)

# =========================================================
# PLOT (beeswarm)
# =========================================================

shap.summary_plot(
    shap_vals,
    X_sample_df,
    max_display=SHAP_MAX_DISPLAY,
    show=False,
    plot_size=(16, 12)
)

ax = plt.gca()

# force y-axis (feature name) font size
ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

# improve colorbar (right side)
fig = plt.gcf()
if len(fig.axes) > 1:
    cb_ax = fig.axes[1]
    cb_ax.set_ylabel("Feature value", fontsize=22, fontweight="bold")
    cb_ax.tick_params(labelsize=20)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.tick_params(axis='y', labelsize=18)
ax.tick_params(axis='x', labelsize=14)

ax.grid(axis='x', color='gray', linestyle=':', alpha=0.25)

lim = np.max(np.abs(shap_vals)) * 1.1
plt.xlim(-lim, lim)

plt.xlabel("SHAP value (impact on model output)", fontsize=20, fontweight="bold", labelpad=15)

plt.subplots_adjust(left=0.35, right=0.95, top=0.95, bottom=0.1)

plt.savefig(f"{results_dir}/SHAP_summary_plot.png", dpi=1500)
plt.show()